# CCE mooring time series — CalCOFI / OceanSITES

This notebook downloads **California Current Ecosystem (CCE) mooring** data from the **OceanSITES THREDDS** server (linked from [CalCOFI Data](https://calcofi.org/data/) and [mooring.ucsd.edu/cce](https://mooring.ucsd.edu/cce/)), merges CTD temperature with nitrate and oxygen, exports **CSV/JSON**, and builds optional **dashboard summary** tables.

---

### Table of contents

| Step | Section | What to run |
|------|---------|-------------|
| 1 | **Setup** | Install packages (`%pip` cell) — once per environment |
| 2 | **Download code** | One code cell: THREDDS helpers + `fetch_cce_mooring_timeseries()` |
| 3 | **Build & save** | Build `df_*` tables and write project CSVs (slow; needs internet) |
| 4 | **Outputs reference** | Read-only: what each file and DataFrame is |
| 5 | **Dashboard mapping** | How mooring data relates to a larger story (optional) |
| 6 | **Dashboard CSVs** | Monthly/annual means, trends, warming spikes |

---

### What gets created (project folder)

| File | Contents |
|------|----------|
| `cce_mooring_timeseries.csv` / `.json` | Main merged time series |
| `cce_thredds_file_catalog.csv` | All NetCDF names from THREDDS |
| `cce_deployment_file_index.csv` | Per-deployment CTD / NO3 / O2 filenames |
| `dashboard_cce_*.csv` | BI-ready aggregates (after Section 6) |

---

### Data notes for readers

- **Sources:** NetCDF on `dods.ndbc.noaa.gov` (OPeNDAP). Not the CalCOFI WordPress API.
- **Moorings:** **CCE1** and **CCE2** only (CCE3/CCE4 not in public THREDDS yet).
- **Temperature** is CTD at depth **nearest to the nitrate sensor** (~40 m), not necessarily sea surface.
- **Policy:** [CalCOFI data usage](https://wp.calcofi.org/wp/data/data-usage-policy/).

In [1]:
# Section 1 — Dependencies (run once per Jupyter kernel)
# Installs into the kernel Python (not necessarily your terminal).
# Recommended: create `.venv`, `pip install -r requirements.txt`, then select that kernel.
%pip install -q xarray netCDF4 pandas requests pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
# =============================================================================
# Section 2 — THREDDS catalog + OPeNDAP merge + fetch
# Run after Section 1. Requires network. Defines all functions used below.
# =============================================================================

import re
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import xarray as xr

THREDDS_CATALOG_XML = (
    "https://dods.ndbc.noaa.gov/thredds/catalog/oceansites/DATA/{mooring}/catalog.xml"
)
OPENDAP_BASE = "https://dods.ndbc.noaa.gov/thredds/dodsC/oceansites/DATA/{mooring}"

NS = {"cat": "http://www.unidata.ucar.edu/namespaces/thredds/InvCatalog/v1.0"}
FILE_RE = re.compile(r"OS_(CCE\d)_(\d+)_D_(CTD|NO3|OXYGEN)\.nc$")


# --- Catalog: list NetCDF files from THREDDS XML ---

@dataclass
class MooringPaths:
    mooring: str
    deployment: int
    ctd: Optional[str] = None
    no3: Optional[str] = None
    oxygen: Optional[str] = None


def fetch_thredds_inventory(mooring: str, timeout: int = 120) -> List[Tuple[str, str]]:
    """Return list of (dataset_name, url_path) from THREDDS catalog XML."""
    url = THREDDS_CATALOG_XML.format(mooring=mooring)
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    out: List[Tuple[str, str]] = []
    for ds in root.findall(".//cat:dataset", NS):
        name = ds.get("name") or ""
        url_path = ds.get("urlPath") or ""
        if name.endswith(".nc") and url_path:
            out.append((name, url_path))
    return out


def group_deployments(inventory: Iterable[Tuple[str, str]]) -> Dict[Tuple[str, int], MooringPaths]:
    """Group NetCDF filenames by (mooring id e.g. CCE1, deployment number)."""
    groups: Dict[Tuple[str, int], MooringPaths] = {}
    for name, _url_path in inventory:
        m = FILE_RE.search(name)
        if not m:
            continue
        mooring_id, dep_str, kind = m.group(1), int(m.group(2)), m.group(3)
        key = (mooring_id, dep_str)
        if key not in groups:
            groups[key] = MooringPaths(mooring=mooring_id, deployment=dep_str)
        g = groups[key]
        if kind == "CTD":
            g.ctd = name
        elif kind == "NO3":
            g.no3 = name
        elif kind == "OXYGEN":
            g.oxygen = name
    return groups


# --- OPeNDAP URL helper ---

def opendap_url(mooring_folder: str, filename: str) -> str:
    return f"{OPENDAP_BASE.format(mooring=mooring_folder)}/{filename}"


def _series_time_var(ds: xr.Dataset) -> str:
    for cand in ("TIME", "time"):
        if cand in ds:
            return cand
    raise ValueError("No TIME/time coordinate found")


# --- Single deployment: align CTD temp + NO3 + optional O2 onto NO3 times ---

def load_merged_deployment(
    mooring_folder: str,
    paths: MooringPaths,
    merge_tolerance: pd.Timedelta = pd.Timedelta("3h"),
    clip_start: Optional[pd.Timestamp] = None,
) -> pd.DataFrame:
    """Merge CTD temperature, nitrate, optional oxygen for one deployment."""
    if not paths.ctd or not paths.no3:
        raise ValueError("CTD and NO3 files are required")

    ds_no3 = xr.open_dataset(opendap_url(mooring_folder, paths.no3), decode_times=True)
    time_name = _series_time_var(ds_no3)
    t_no3 = pd.to_datetime(ds_no3[time_name].values)
    if clip_start is not None:
        last = pd.Timestamp(t_no3.max())
        if last.tzinfo is None:
            last = last.tz_localize("UTC")
        if last < clip_start:
            ds_no3.close()
            return pd.DataFrame()

    ds_ctd = xr.open_dataset(opendap_url(mooring_folder, paths.ctd), decode_times=True)
    ctd_time = _series_time_var(ds_ctd)

    no3_depth = float(np.asarray(ds_no3.DEPTH.values).ravel()[0])
    depth_ctd = np.asarray(ds_ctd.DEPTH.values).ravel()
    idx_ctd = int(np.argmin(np.abs(depth_ctd - no3_depth)))

    t_n = t_no3
    nvals = ds_no3["NO3"]
    if "DEPTH" in nvals.dims:
        nvals = nvals.isel(DEPTH=0)
    nvals = np.asarray(nvals.values).ravel()
    df_n = pd.DataFrame({time_name.lower(): t_n, "nitrate_umol_l": nvals})

    t_c = pd.to_datetime(ds_ctd[ctd_time].values)
    temp_da = ds_ctd["TEMP"]
    if "DEPTH" in temp_da.dims:
        temp_da = temp_da.isel(DEPTH=idx_ctd)
    temp = np.asarray(temp_da.values).ravel()
    df_c = pd.DataFrame({time_name.lower(): t_c, "temperature_c": temp})

    lat = float(np.asarray(ds_no3["LATITUDE"].values).ravel()[0])
    lon = float(np.asarray(ds_no3["LONGITUDE"].values).ravel()[0])

    time_col = time_name.lower()
    master = df_n.sort_values(time_col).reset_index(drop=True)
    df_c = df_c.sort_values(time_col)

    out = pd.merge_asof(
        master,
        df_c,
        on=time_col,
        direction="nearest",
        tolerance=merge_tolerance,
    )

    ds_o2 = None
    if paths.oxygen:
        ds_o2 = xr.open_dataset(opendap_url(mooring_folder, paths.oxygen), decode_times=True)
        o2_time = _series_time_var(ds_o2)
        depth_o2 = np.asarray(ds_o2.DEPTH.values).ravel()
        idx_o2 = int(np.argmin(np.abs(depth_o2 - no3_depth)))
        t_o = pd.to_datetime(ds_o2[o2_time].values)
        oxy = ds_o2["DOX2"]
        if "DEPTH" in oxy.dims:
            oxy = oxy.isel(DEPTH=idx_o2)
        oxy = np.asarray(oxy.values).ravel()
        df_o = pd.DataFrame({time_col: t_o, "oxygen_umol_kg": oxy}).sort_values(time_col)
        out = pd.merge_asof(
            out,
            df_o,
            on=time_col,
            direction="nearest",
            tolerance=merge_tolerance,
        )
    else:
        out["oxygen_umol_kg"] = np.nan

    out.rename(columns={time_col: "time_utc"}, inplace=True)
    out["latitude"] = lat
    out["longitude"] = lon
    out["reference_depth_m"] = no3_depth
    out["mooring"] = paths.mooring
    out["deployment"] = paths.deployment
    out["mooring_folder"] = mooring_folder
    out["source_file_ctd"] = paths.ctd
    out["source_file_no3"] = paths.no3
    out["source_file_oxygen"] = paths.oxygen

    ds_no3.close()
    ds_ctd.close()
    if ds_o2 is not None:
        ds_o2.close()

    return out


# --- All deployments: CCE1/CCE2, optional year clip ---

def fetch_cce_mooring_timeseries(
    years: int = 10,
    mooring_folders: Optional[List[str]] = None,
    reference_time: Optional[datetime] = None,
    merge_tolerance: pd.Timedelta = pd.Timedelta("3h"),
) -> pd.DataFrame:
    """
    Pull merged mooring rows for all deployments that overlap the last `years` years.

    Mooring folder names match THREDDS paths (e.g. 'CCE1', 'CCE2').
    """
    if reference_time is None:
        reference_time = datetime.now(timezone.utc)
    start = reference_time - timedelta(days=365 * years)

    if mooring_folders is None:
        mooring_folders = ["CCE1", "CCE2"]

    frames: List[pd.DataFrame] = []
    for folder in mooring_folders:
        inv = fetch_thredds_inventory(folder)
        groups = group_deployments(inv)
        for (_mid, _dep), paths in sorted(groups.items(), key=lambda x: (x[0][0], x[0][1])):
            if not (paths.ctd and paths.no3):
                continue
            try:
                df = load_merged_deployment(
                    folder,
                    paths,
                    merge_tolerance=merge_tolerance,
                    clip_start=pd.Timestamp(start),
                )
            except Exception as exc:
                print(f"Skip {folder} deployment {paths.deployment}: {exc}")
                continue
            if df.empty:
                continue
            df["time_utc"] = pd.to_datetime(df["time_utc"], utc=True)
            df = df[df["time_utc"] >= pd.Timestamp(start)]
            if df.empty:
                continue
            frames.append(df)

    if not frames:
        return pd.DataFrame()

    out = pd.concat(frames, ignore_index=True)
    out.sort_values(["mooring", "deployment", "time_utc"], inplace=True)
    out.reset_index(drop=True, inplace=True)
    return out


# --- Export ---

def dataframe_to_json_records(df: pd.DataFrame, path: str) -> None:
    """Write JSON array (records orientation); timestamps as ISO8601 strings."""
    tmp = df.copy()
    if "time_utc" in tmp.columns:
        tmp["time_utc"] = tmp["time_utc"].dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
    tmp.to_json(path, orient="records", indent=2)


print("Ready: run Section 3 to build DataFrames and save CSV files.")

Ready: run Section 3 to build DataFrames and save CSV files.


### Section 3 — Build tables and save files

Run the next code cell after Section 2. It may take **several minutes** (many NetCDF reads over the internet).

**Python variables created:**

| Variable | Description |
|----------|-------------|
| `df_cce_mooring_timeseries` | **Main table:** one row per **nitrate** time; CTD **temperature** (depth matched to nitrate), **nitrate**, **dissolved oxygen**; lat/lon; `deployment`; source filenames. |
| `df_cce_thredds_file_catalog` | Every `.nc` in the THREDDS catalog (CCE1 + CCE2), including ADCP, met, etc. |
| `df_cce_deployment_file_index` | One row per deployment: which CTD / NO3 / O2 files exist. |
| `DATAFRAME_REFERENCE` | Dict of short string descriptions (same info as this table). |

**Files written:** `cce_mooring_timeseries.csv`, `.json`, optional `.parquet`; `cce_thredds_file_catalog.csv`; `cce_deployment_file_index.csv`.

---

### Section 4 — Column reference (`df_cce_mooring_timeseries`)

| Column | Meaning |
|--------|---------|
| `time_utc` | Timestamp (UTC) |
| `temperature_c` | CTD sea temperature (°C), depth nearest nitrate |
| `nitrate_umol_l` | Nitrate (µmol L⁻¹) |
| `oxygen_umol_kg` | Dissolved oxygen DOX2 (µmol kg⁻¹) |
| `latitude`, `longitude` | Mooring position |
| `reference_depth_m` | Nitrate sensor depth (~m) |
| `mooring`, `deployment`, `mooring_folder` | IDs |
| `source_file_*` | Provenance NetCDF names |

In [3]:
# Section 3 — Build DataFrames, print summary, write CSV/JSON/Parquet

MOORINGS = ["CCE1", "CCE2"]
YEARS = 10

# 1) THREDDS catalog: all NetCDF entries (metadata only; fast)
_rows_cat = []
for _folder in MOORINGS:
    for _name, _path in fetch_thredds_inventory(_folder):
        _rows_cat.append(
            {
                "mooring_folder": _folder,
                "netcdf_filename": _name,
                "thredds_url_path": _path,
            }
        )
df_cce_thredds_file_catalog = pd.DataFrame(_rows_cat)

# 2) Deployment index: which instrument files exist per deployment (from catalog above; no extra HTTP)
_rows_dep = []
for _folder in MOORINGS:
    _sub = df_cce_thredds_file_catalog[
        df_cce_thredds_file_catalog["mooring_folder"] == _folder
    ]
    _inv = list(zip(_sub["netcdf_filename"], _sub["thredds_url_path"]))
    _groups = group_deployments(_inv)
    for (_m, _dep), _p in sorted(_groups.items(), key=lambda x: (x[0][0], x[0][1])):
        _rows_dep.append(
            {
                "mooring_id": _m,
                "deployment_number": _dep,
                "mooring_folder": _folder,
                "has_ctd_file": _p.ctd is not None,
                "has_no3_file": _p.no3 is not None,
                "has_oxygen_file": _p.oxygen is not None,
                "file_ctd": _p.ctd,
                "file_no3": _p.no3,
                "file_oxygen": _p.oxygen,
            }
        )
df_cce_deployment_file_index = pd.DataFrame(_rows_dep)

# 3) Primary merged time series (slow: opens NetCDF via OPeNDAP)
df_cce_mooring_timeseries = fetch_cce_mooring_timeseries(
    years=YEARS,
    mooring_folders=MOORINGS,
    # reference_time=datetime(2026, 4, 18, tzinfo=timezone.utc),
)

DATAFRAME_REFERENCE = {
    "df_cce_mooring_timeseries": (
        "Merged CCE mooring observations: one row per nitrate timestamp; "
        "temperature (CTD, depth matched to nitrate), nitrate (µmol/L), oxygen DOX2 (µmol/kg); "
        "columns include mooring, deployment, lat/lon, provenance filenames."
    ),
    "df_cce_thredds_file_catalog": (
        "Full THREDDS listing for CCE1/CCE2: every NetCDF filename and urlPath from catalog XML."
    ),
    "df_cce_deployment_file_index": (
        "One row per mooring deployment with flags and CTD/NO3/OXYGEN filenames used for merging."
    ),
}

for _name, _desc in DATAFRAME_REFERENCE.items():
    print(f"\n=== {_name} ===\n{_desc}")

print("\n--- shapes ---")
print("df_cce_mooring_timeseries:", df_cce_mooring_timeseries.shape)
print("df_cce_thredds_file_catalog:", df_cce_thredds_file_catalog.shape)
print("df_cce_deployment_file_index:", df_cce_deployment_file_index.shape)

print("\n--- df_cce_mooring_timeseries (head) ---")
print(df_cce_mooring_timeseries.head())

# --- Save for use outside this notebook ---
OUT_DIR = "."  # set to a folder path if you prefer
PATH_MERGED_CSV = f"{OUT_DIR}/cce_mooring_timeseries.csv"
PATH_MERGED_JSON = f"{OUT_DIR}/cce_mooring_timeseries.json"
PATH_CATALOG_CSV = f"{OUT_DIR}/cce_thredds_file_catalog.csv"
PATH_DEPLOY_INDEX_CSV = f"{OUT_DIR}/cce_deployment_file_index.csv"

df_cce_mooring_timeseries.to_csv(PATH_MERGED_CSV, index=False)
dataframe_to_json_records(df_cce_mooring_timeseries, PATH_MERGED_JSON)
df_cce_thredds_file_catalog.to_csv(PATH_CATALOG_CSV, index=False)
df_cce_deployment_file_index.to_csv(PATH_DEPLOY_INDEX_CSV, index=False)

# Optional: Parquet (single file, preserves dtypes; requires pyarrow or fastparquet)
try:
    df_cce_mooring_timeseries.to_parquet(f"{OUT_DIR}/cce_mooring_timeseries.parquet", index=False)
    print("\nAlso wrote: cce_mooring_timeseries.parquet")
except Exception as _e:
    print("\nParquet not written (install pyarrow in the kernel if you want it):", _e)

print("\nWrote:", PATH_MERGED_CSV, PATH_MERGED_JSON, PATH_CATALOG_CSV, PATH_DEPLOY_INDEX_CSV)

# Backwards-compatible alias used in earlier notebook versions
df = df_cce_mooring_timeseries


=== df_cce_mooring_timeseries ===
Merged CCE mooring observations: one row per nitrate timestamp; temperature (CTD, depth matched to nitrate), nitrate (µmol/L), oxygen DOX2 (µmol/kg); columns include mooring, deployment, lat/lon, provenance filenames.

=== df_cce_thredds_file_catalog ===
Full THREDDS listing for CCE1/CCE2: every NetCDF filename and urlPath from catalog XML.

=== df_cce_deployment_file_index ===
One row per mooring deployment with flags and CTD/NO3/OXYGEN filenames used for merging.

--- shapes ---
df_cce_mooring_timeseries: (130087, 13)
df_cce_thredds_file_catalog: (266, 3)
df_cce_deployment_file_index: (30, 9)

--- df_cce_mooring_timeseries (head) ---
                             time_utc  nitrate_umol_l  temperature_c  \
0 2016-04-21 17:00:00.000003328+00:00        0.637643      14.984310   
1           2016-04-21 18:00:00+00:00        0.565154      14.983310   
2 2016-04-21 18:59:59.999996672+00:00        0.424164      15.098009   
3 2016-04-21 20:00:00.000003328+0

### Section 5 — Dashboard story: what this data supports

`df_cce_mooring_timeseries` / `cce_mooring_timeseries.csv` covers **mooring temperature + chemistry** only. For larvae, maps, or ML you must add other datasets. Use this table to plan integrations.

| Story beat | Dashboard viz | From your mooring export | Still needed |
|------------|----------------|---------------------------|--------------|
| **1. Trigger — warming & chemistry** | Heat map of **trends** + acidification markers | **Trend / anomaly inputs:** `temperature_c`, `nitrate_umol_l`, `oxygen_umol_kg` by `time_utc`, `mooring`, `latitude`, `longitude`. Use **monthly/annual** aggregates + linear trend (slope per mooring). **Caveat:** temperature here is **CTD at ~40 m** (matched to nitrate), not literal **sea-surface**; for surface SST use shallowest CTD bin or mooring met data in THREDDS. **No pH/pCO₂/DIC** in this CSV — add `*_P_PH*.nc` / carbonate NetCDF from [THREDDS CCE catalogs](https://dods.ndbc.noaa.gov/thredds/catalog/oceansites/DATA/CCE1/catalog.html) or CalCOFI **bottle/DIC** ([CalCOFI Data](https://calcofi.org/data/)). |
| **2. Biological toll — larvae vs warming** | Decline chart: larvae vs warming spikes | **Join key:** `year`, `year_month`, or `season` from mooring side; define **warming spikes** (e.g. monthly anomaly > 1.5σ). | Download **CalCOFI Fish Eggs & Larvae** (or ichthyoplankton) time series from the portal; aggregate **anchovy** (or target taxon) abundance/catch per cruise or region; merge on time/region. |
| **3. Ecological impact — vanishing biodiversity** | Map: historic CalCOFI zones vs current iNat | Mooring gives **fixed point** lat/lon per deployment — useful as reference points, not species map. | **iNaturalist** [API](https://www.inaturalist.org/pages/api+reference) or export; **historic** CalCOFI station grid / regions as polygons; compare anchovy (or clupeid) observation density **then vs now**. |
| **4. Future — clean water probability** | ML slider 2030–2050 | Use merged chemistry + temp as **historical features** for training if you label “good” water years. | **Scenario inputs** (nutrient reduction %) are assumptions; need a **model** (e.g. regression/classifier or simple scenario rules) trained on features + outcome. Not derivable from one CSV alone. |

### Section 6 — Dashboard-ready CSVs (next cell)

Builds monthly/annual aggregates, trend slopes, and warming-spike flags → `dashboard_cce_*.csv` for Tableau, Observable, etc.

In [4]:
# Section 6 — Dashboard aggregates (run after Section 3, or set _path to a saved CSV)

_path = "cce_mooring_timeseries.csv"
try:
    d = df_cce_mooring_timeseries.copy()
except NameError:
    d = pd.read_csv(_path)
    d["time_utc"] = pd.to_datetime(d["time_utc"], utc=True, format="mixed")

d = d.sort_values(["mooring", "deployment", "time_utc"])

# Monthly means (heatmap / time series: mooring × year-month)
d["year_month"] = d["time_utc"].dt.strftime("%Y-%m")
d["year"] = d["time_utc"].dt.year
monthly = (
    d.groupby(["mooring", "mooring_folder", "latitude", "longitude", "year_month"], as_index=False)
    .agg(
        temperature_c_mean=("temperature_c", "mean"),
        nitrate_umol_l_mean=("nitrate_umol_l", "mean"),
        oxygen_umol_kg_mean=("oxygen_umol_kg", "mean"),
        n_obs=("time_utc", "count"),
    )
)

# Annual means (long-term trend charts)
annual = (
    d.groupby(["mooring", "mooring_folder", "latitude", "longitude", "year"], as_index=False)
    .agg(
        temperature_c_mean=("temperature_c", "mean"),
        nitrate_umol_l_mean=("nitrate_umol_l", "mean"),
        oxygen_umol_kg_mean=("oxygen_umol_kg", "mean"),
    )
)


def linear_slope(years: np.ndarray, values: np.ndarray) -> float:
    mask = np.isfinite(values)
    if mask.sum() < 3:
        return np.nan
    x, yv = years[mask].astype(float), values[mask]
    return float(np.polyfit(x, yv, 1)[0])


# 20-year-style trend: slope of annual mean temp per mooring (requires enough years in file)
trend_rows = []
for (m, lat, lon), g in annual.groupby(["mooring", "latitude", "longitude"]):
    g = g.sort_values("year")
    trend_rows.append(
        {
            "mooring": m,
            "latitude": lat,
            "longitude": lon,
            "year_span_start": int(g["year"].min()),
            "year_span_end": int(g["year"].max()),
            "n_years": len(g),
            "temp_trend_c_per_year": linear_slope(g["year"].values, g["temperature_c_mean"].values),
            "nitrate_trend_umol_l_per_year": linear_slope(g["year"].values, g["nitrate_umol_l_mean"].values),
        }
    )
df_dashboard_cce_trends = pd.DataFrame(trend_rows)

# "Warming spike" flag: monthly temp anomaly vs long-run monthly climatology (same calendar month)
d["calendar_month"] = d["time_utc"].dt.month
clim = (
    d.groupby(["mooring", "calendar_month"])["temperature_c"]
    .mean()
    .rename("temp_clim")
    .reset_index()
)
d = d.merge(clim, on=["mooring", "calendar_month"], how="left")
d["temp_anomaly_c"] = d["temperature_c"] - d["temp_clim"]
monthly_std = d.groupby(["mooring", "calendar_month"])["temperature_c"].transform("std")
d["warming_spike"] = d["temp_anomaly_c"] > 1.5 * monthly_std.replace(0, np.nan)

df_dashboard_cce_spikes = d[
    [
        "time_utc",
        "year",
        "year_month",
        "mooring",
        "latitude",
        "longitude",
        "temperature_c",
        "temp_anomaly_c",
        "warming_spike",
        "nitrate_umol_l",
        "oxygen_umol_kg",
    ]
].copy()

monthly.to_csv("dashboard_cce_monthly_means.csv", index=False)
annual.to_csv("dashboard_cce_annual_means.csv", index=False)
df_dashboard_cce_trends.to_csv("dashboard_cce_trend_slopes.csv", index=False)
df_dashboard_cce_spikes.to_csv("dashboard_cce_warming_spikes.csv", index=False)

print("Saved:")
print("  dashboard_cce_monthly_means.csv   — heat map / ribbon: mooring × year_month")
print("  dashboard_cce_annual_means.csv    — line charts: year × mooring")
print("  dashboard_cce_trend_slopes.csv    — trend slopes °C/yr & nitrate/yr per mooring")
print("  dashboard_cce_warming_spikes.csv  — join to larvae on year_month for panel 2")
print("\nDataFrames: monthly, annual, df_dashboard_cce_trends, df_dashboard_cce_spikes")


Saved:
  dashboard_cce_monthly_means.csv   — heat map / ribbon: mooring × year_month
  dashboard_cce_annual_means.csv    — line charts: year × mooring
  dashboard_cce_trend_slopes.csv    — trend slopes °C/yr & nitrate/yr per mooring
  dashboard_cce_warming_spikes.csv  — join to larvae on year_month for panel 2

DataFrames: monthly, annual, df_dashboard_cce_trends, df_dashboard_cce_spikes


In [5]:
pip install pyinaturalist pandas

Note: you may need to restart the kernel to use updated packages.


In [6]:
from pyinaturalist import get_observations
import pandas as pd

# 1. Fetch Northern Anchovy data around the CCE area (Southern California coast)
print("Fetching iNaturalist data...")
response = get_observations(
    taxon_name='Engraulis mordax', 
    nelat=35.0,   # North East Latitude
    nelng=-117.0, # North East Longitude
    swlat=32.0,   # South West Latitude
    swlng=-125.0, # South West Longitude
    d1='2014-01-01', # Matching your "last10y" timeframe
    d2='2024-12-31', 
    per_page=200  # Max results per API call
)

# 2. Extract the coordinates and dates into a DataFrame
obs_data = []
for obs in response.get('results', []):
    if obs.get('location') and obs.get('observed_on'):
        obs_data.append({
            'date': obs.get('observed_on'),
            'lat': obs['location'][0],
            'lon': obs['location'][1],
        })

df_anchovy = pd.DataFrame(obs_data)
# Clean up datetime formats
df_anchovy['date'] = pd.to_datetime(df_anchovy['date'], utc=True).dt.tz_localize(None) 
df_anchovy['year_month'] = df_anchovy['date'].dt.to_period('M')

# 3. Aggregate biological data: Count sightings per month
df_anchovy_monthly = df_anchovy.groupby('year_month').size().reset_index(name='anchovy_sightings')

# 4. Load your existing CCE physical data
print("Loading physical data...")
df_physical = pd.read_csv('dashboard_cce_monthly_means.csv')

# Ensure both year_month columns are strings so they merge perfectly (e.g., '2020-01')
df_anchovy_monthly['year_month'] = df_anchovy_monthly['year_month'].astype(str)
df_physical['year_month'] = df_physical['year_month'].astype(str)

# 5. Merge Biological (Anchovy) and Physical (Temp, Nitrate, Oxygen) data
df_merged = pd.merge(df_physical, df_anchovy_monthly, on='year_month', how='left')

# If there were no sightings in a month, set the count to 0 instead of NaN
df_merged['anchovy_sightings'] = df_merged['anchovy_sightings'].fillna(0)

# Clean up the output
df_merged.to_csv('dashboard_ecological_impact.csv', index=False)

print(f"Merge complete! {len(df_merged)} months of data saved to dashboard_ecological_impact.csv")

Fetching iNaturalist data...
Loading physical data...
Merge complete! 203 months of data saved to dashboard_ecological_impact.csv


In [7]:
df_physical = pd.read_csv('dashboard_cce_monthly_means.csv')
print(df_physical.columns) # Find the exact name of your time column

Index(['mooring', 'mooring_folder', 'latitude', 'longitude', 'year_month',
       'temperature_c_mean', 'nitrate_umol_l_mean', 'oxygen_umol_kg_mean',
       'n_obs'],
      dtype='object')


In [8]:
pip install dash

Note: you may need to restart the kernel to use updated packages.


In [9]:
!python app.py

Dash is running on http://127.0.0.1:8050/

 * Serving Flask app 'app'
 * Debug mode: on
^C


In [10]:
!pip install dash scikit-learn joblib

In [11]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import joblib

# 1. Load your merged data
df = pd.read_csv('dashboard_ecological_impact.csv')

# 2. Create the "Clean Water" target (1 = Clean, 0 = Degraded)
med_o2 = df['oxygen_umol_kg_mean'].median()
med_no3 = df['nitrate_umol_l_mean'].median()
df['clean_water'] = ((df['oxygen_umol_kg_mean'] > med_o2) & (df['nitrate_umol_l_mean'] < med_no3)).astype(int)

# 3. Train the Model
features = ['temperature_c_mean', 'nitrate_umol_l_mean']
X = df[features].fillna(df[features].mean())
y = df['clean_water']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

# 4. Save Model
joblib.dump(rf, 'clean_water_model.pkl')
print("Model saved to clean_water_model.pkl! You can now run the dashboard cell.")

Model saved to clean_water_model.pkl! You can now run the dashboard cell.


In [12]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd
import numpy as np
import joblib

# 1. Load the trained model 
# (Make sure you ran the training script from earlier to generate this .pkl file)
model = joblib.load('clean_water_model.pkl')

# Define baselines
BASE_TEMP = 15.0  
BASE_NITRATE = 10.0 

# 2. Initialize App
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H2("Future Projection: Clean Water Probability (2026-2050)", style={'textAlign': 'center', 'fontFamily': 'sans-serif'}),
    
    html.Div([
        html.Label("Nitrate Reduction % (Policy Intervention):", style={'fontWeight': 'bold'}),
        dcc.Slider(
            id='nitrate-slider', 
            min=0, max=100, value=20, 
            marks={i: f'{i}%' for i in range(0, 101, 20)}
        ),
        
        html.Br(),
        
        html.Label("Sea Surface Temp Anomaly °C (Climate Scenario):", style={'fontWeight': 'bold'}),
        dcc.Slider(
            id='temp-slider', 
            min=0.0, max=3.0, value=1.0, step=0.1, 
            marks={i/10: f'+{i/10}°C' for i in range(0, 31, 5)}
        ),
    ], style={'width': '80%', 'margin': 'auto', 'paddingBottom': '30px', 'fontFamily': 'sans-serif'}),
    
    dcc.Graph(id='projection-graph')
])

# 3. Model Inference Logic
@app.callback(
    Output('projection-graph', 'figure'),
    [Input('nitrate-slider', 'value'),
     Input('temp-slider', 'value')]
)
def update_projection(nitrate_reduction, temp_anomaly):
    years = np.arange(2026, 2051)
    
    # Simulate feature changes over time based on sliders
    temp_trend = np.linspace(BASE_TEMP, BASE_TEMP + temp_anomaly, len(years))
    
    target_nitrate = BASE_NITRATE * (1 - nitrate_reduction / 100)
    nitrate_trend = np.linspace(BASE_NITRATE, target_nitrate, len(years))
    
    # Create the future dataframe
    future_df = pd.DataFrame({
        'temperature_c_mean': temp_trend,
        'nitrate_umol_l_mean': nitrate_trend
    })
    
    # Predict probabilities (prob_class_1 = Clean Water)
    probs = model.predict_proba(future_df)[:, 1]
    
    # Plot the results
    plot_df = pd.DataFrame({'Year': years, 'Probability': probs * 100})
    fig = px.line(plot_df, x='Year', y='Probability', 
                  range_y=[0, 100],
                  labels={'Probability': 'Probability of Clean Water (%)'})
    
    fig.update_traces(line=dict(width=4, color='#2ca02c'))
    fig.update_layout(margin={"r":40,"t":40,"l":40,"b":40})
    
    return fig

# 4. Run the app in Jupyter Mode
if __name__ == '__main__':
    # This will print a clickable link right below the cell
    app.run(jupyter_mode="external") 
    
    # Alternatively, use jupyter_mode="inline" to embed the dashboard directly inside the notebook!

Dash app running on http://127.0.0.1:8050/


In [13]:
joblib.dump(rf, '/Users/kanglee/Downloads/clean_water_model.pkl')
print("Saved directly to downloads!")

Saved directly to downloads!
